In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_scenario

In [2]:
location = "india"
vehicle = "rice"
# Which result pathways this (location, vehicle) combo has. Snakemake passes
# these from config_utils.get_combo_pathways, the same function that decides
# which inputs the rule declares. A pathway the combo lacks contributes zeros,
# built from the template combo (india/rice). Never branch on whether a file
# exists: a job can start before the file is built, and a file missing by
# mistake would silently become zeros.
has_simulations = True
has_anemia_model = True
has_ntd_model = True

In [3]:
# Parameters
location = "nigeria"
vehicle = "wheat"
has_simulations = True
has_anemia_model = True
has_ntd_model = True


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if has_simulations:
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,1,baseline,0,141,35.416449
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,2,baseline,0,141,36.912919
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,3,baseline,0,141,37.380566
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,4,baseline,0,141,35.291744
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,5,baseline,0,141,30.771158
...,...,...,...,...,...,...,...,...,...,...,...
1079995,person_time,impairment,anemia,severe,95_plus,severe,1,zero,0,129,0.000000
1079996,person_time,impairment,anemia,severe,95_plus,severe,2,zero,0,129,0.000000
1079997,person_time,impairment,anemia,severe,95_plus,severe,3,zero,0,129,0.000000
1079998,person_time,impairment,anemia,severe,95_plus,severe,4,zero,0,129,0.000000


In [6]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [7]:
pregnancy_person_time_anemia.sub_entity.value_counts()

not_anemic    270000
mild          270000
moderate      270000
severe        270000
Name: sub_entity, dtype: int64

In [8]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario      wealth_quintile
baseline      1                  2.485855e+06
              2                  2.580952e+06
              3                  2.255781e+06
              4                  2.010391e+06
              5                  1.527952e+06
intervention  1                  2.485857e+06
              2                  2.580954e+06
              3                  2.255783e+06
              4                  2.010393e+06
              5                  1.527954e+06
zero          1                  2.485850e+06
              2                  2.580945e+06
              3                  2.255775e+06
              4                  2.010383e+06
              5                  1.527946e+06
Name: value, dtype: float64

In [9]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario      wealth_quintile
baseline      1                  1.312310e+06
              2                  1.328116e+06
              3                  1.126331e+06
              4                  1.020841e+06
              5                  6.935400e+05
intervention  1                  1.305068e+06
              2                  1.319054e+06
              3                  1.115259e+06
              4                  1.011614e+06
              5                  6.833713e+05
zero          1                  1.332873e+06
              2                  1.354361e+06
              3                  1.158762e+06
              4                  1.047664e+06
              5                  7.244389e+05
Name: value, dtype: float64

In [10]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario      wealth_quintile
baseline      1                  0.527911
              2                  0.514584
              3                  0.499308
              4                  0.507782
              5                  0.453902
intervention  1                  0.524997
              2                  0.511072
              3                  0.494400
              4                  0.503192
              5                  0.447246
zero          1                  0.536184
              2                  0.524754
              3                  0.513687
              4                  0.521127
              5                  0.474126
Name: value, dtype: float64

In [11]:
path = f"./results/{location}/{vehicle}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [12]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,1,18793.149621
1,Female,0.0,0.019178,not_pregnant,2,17830.620200
2,Female,0.0,0.019178,not_pregnant,3,16129.662368
3,Female,0.0,0.019178,not_pregnant,4,14114.028154
4,Female,0.0,0.019178,not_pregnant,5,12208.445301
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,1,4849.649774
281,Male,95.0,125.000000,not_pregnant,2,4371.181536
282,Male,95.0,125.000000,not_pregnant,3,4569.918352
283,Male,95.0,125.000000,not_pregnant,4,4550.906874


In [13]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
1    2.306716e+06
2    2.394711e+06
3    2.088829e+06
4    1.862288e+06
5    1.411894e+06
Name: value, dtype: float64

In [14]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.217740e+06
              2                  1.232280e+06
              3                  1.042970e+06
              4                  9.456368e+05
              5                  6.408611e+05
intervention  1                  1.211020e+06
              2                  1.223870e+06
              3                  1.032717e+06
              4                  9.370881e+05
              5                  6.314641e+05
zero          1                  1.236824e+06
              2                  1.256634e+06
              3                  1.073004e+06
              4                  9.704876e+05
              5                  6.694156e+05
Name: value, dtype: float64

In [15]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if has_simulations:
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet"
        ).assign(value=0),
        scenarios,
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,1,baseline,0,141,0.0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,2,baseline,0,141,0.0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,3,baseline,0,141,0.0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,4,baseline,0,141,0.0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,5,baseline,0,141,0.0
...,...,...,...,...,...,...,...,...,...,...,...
539995,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,1,zero,0,129,0.0
539996,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,2,zero,0,129,0.0
539997,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,3,zero,0,129,0.0
539998,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,4,zero,0,129,0.0


In [16]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['susceptible_to_maternal_disorders_to_maternal_disorders', 'maternal_disorders_to_recovered_from_maternal_disorders'], dtype='object')

In [17]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.561085e+06
              2                  2.216806e+06
              3                  2.114086e+06
              4                  1.925897e+06
              5                  1.405792e+06
intervention  1                  2.556784e+06
              2                  2.211409e+06
              3                  2.107756e+06
              4                  1.920088e+06
              5                  1.400473e+06
zero          1                  2.573912e+06
              2                  2.233270e+06
              3                  2.133374e+06
              4                  1.943164e+06
              5                  1.422365e+06
Name: value, dtype: float64

In [18]:
path = (
    f"./results/{location}/{vehicle}/maternal_disorders_incident_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [19]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_scenario below, so we'd need to
# change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if has_simulations:
    neonatal_deaths = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    neonatal_deaths = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,deaths,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,baseline,0,81,146.630818
1,deaths,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,baseline,0,81,172.698519
2,deaths,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,baseline,0,81,164.552362
3,deaths,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,baseline,0,81,148.260049
4,deaths,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,baseline,0,81,109.158498
...,...,...,...,...,...,...,...,...,...,...,...,...
23995,deaths,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,71,166.181593
23996,deaths,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,71,149.889280
23997,deaths,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,71,164.552362
23998,deaths,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,71,105.900035


In [20]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario      wealth_quintile
baseline      1                  172903.801774
              2                  179047.633036
              3                  158763.703254
              4                  140314.287924
              5                  106730.942976
intervention  1                  172897.284849
              2                  179023.194567
              3                  158736.006322
              4                  140286.590992
              5                  106701.616812
zero          1                  172934.757169
              2                  179103.026901
              3                  158837.018663
              4                  140390.861795
              5                  106804.258385
Name: value, dtype: float64

In [21]:
path = f"./results/{location}/{vehicle}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [22]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/anemia_cases.parquet"
if has_anemia_model:
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/anemia_cases.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,15351.030080,zero
1,Female,0.0,0.019178,2,14304.542336,zero
2,Female,0.0,0.019178,3,12827.952080,zero
3,Female,0.0,0.019178,4,11083.389649,zero
4,Female,0.0,0.019178,5,9164.329935,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,3617.728615,intervention
746,Male,95.0,125.000000,2,3294.468794,intervention
747,Male,95.0,125.000000,3,3487.652095,intervention
748,Male,95.0,125.000000,4,3482.300754,intervention


In [23]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.406854e+07
              2                  2.362762e+07
              3                  2.189333e+07
              4                  2.066525e+07
              5                  1.941819e+07
intervention  1                  2.395101e+07
              2                  2.346458e+07
              3                  2.168197e+07
              4                  2.043163e+07
              5                  1.916840e+07
zero          1                  2.439851e+07
              2                  2.408536e+07
              3                  2.248670e+07
              4                  2.132115e+07
              5                  2.011946e+07
Name: value, dtype: float64

In [24]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.528629e+07
              2                  2.485990e+07
              3                  2.293630e+07
              4                  2.161089e+07
              5                  2.005905e+07
intervention  1                  2.516203e+07
              2                  2.468845e+07
              3                  2.271469e+07
              4                  2.136872e+07
              5                  1.979987e+07
zero          1                  2.563534e+07
              2                  2.534200e+07
              3                  2.355971e+07
              4                  2.229164e+07
              5                  2.078888e+07
Name: value, dtype: float64

In [25]:
path = f"./results/{location}/{vehicle}/prevalent_anemia_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [26]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
if has_ntd_model:
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ntd_cases_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario      wealth_quintile
zero          1                  15469.942748
              2                  17341.890303
              3                  20540.062077
              4                  23326.520697
              5                  32674.251808
baseline      1                  12734.955657
              2                  13022.284791
              3                  11788.751094
              4                  10449.533117
              5                   9178.019131
intervention  1                  12012.214448
              2                  12022.785954
              3                  10384.955128
              4                   8943.846405
              5                   7606.259787
Name: value, dtype: float64

In [27]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)